<a href="https://colab.research.google.com/github/LegalIntermediaSL/Physics/blob/main/Simulaciones_Jupyter/Mecanica_Cuantica/Mecanica_Cuantica_Heisenberg.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



# Investigación Avanzada: Mecanica Cuantica Heisenberg

Simulación numérica de vanguardia correspondiente a los Problemas Abiertos y Fronteras de Investigación (Nivel Doctorado).

In [ ]:
%matplotlib inline
import numpy as np
import scipy.sparse as sparse
import scipy.sparse.linalg as sla
import matplotlib.pyplot as plt

def kron_pad(op, i, N):
    I = sparse.eye(2, format='csr')
    res = sparse.eye(1, format='csr')
    for j in range(N):
        if j == i:
            res = sparse.kron(res, op)
        else:
            res = sparse.kron(res, I)
    return res

def main():
    # 1D Heisenberg Spin Chain Entanglement
    N = 8 # Number of spins
    J = 1.0 # Coupling
    
    sx = sparse.csr_matrix([[0, 1], [1, 0]])
    sy = sparse.csr_matrix([[0, -1j], [1j, 0]])
    sz = sparse.csr_matrix([[1, 0], [0, -1]])
    
    H = sparse.csr_matrix((2**N, 2**N), dtype=complex)
    
    for i in range(N-1):
        H += J * (kron_pad(sx, i, N) @ kron_pad(sx, i+1, N) +
                  kron_pad(sy, i, N) @ kron_pad(sy, i+1, N) +
                  kron_pad(sz, i, N) @ kron_pad(sz, i+1, N))
                  
    # Periodic boundary
    H += J * (kron_pad(sx, N-1, N) @ kron_pad(sx, 0, N) +
              kron_pad(sy, N-1, N) @ kron_pad(sy, 0, N) +
              kron_pad(sz, N-1, N) @ kron_pad(sz, 0, N))
              
    evals, evecs = sla.eigsh(H, k=1, which='SA')
    gs = evecs[:, 0]
    
    # Calculate entanglement entropy for bipartite split (N/2)
    rho = np.outer(gs, gs.conj())
    rho_tensor = rho.reshape((2**(N//2), 2**(N - N//2), 2**(N//2), 2**(N - N//2)))
    # Partial trace over half system
    rho_A = np.trace(rho_tensor, axis1=1, axis2=3)
    
    eigvals_rho_A = np.linalg.eigvalsh(rho_A)
    eigvals_rho_A = eigvals_rho_A[eigvals_rho_A > 1e-12] # filter zero eigenvalues
    entropy = -np.sum(eigvals_rho_A * np.log(eigvals_rho_A))
    
    print(f"Ground state energy: {evals[0].real:.4f}")
    print(f"Entanglement entropy (half-chain): {entropy.real:.4f}")
    
    plt.figure(figsize=(6, 4))
    plt.bar(range(len(eigvals_rho_A)), eigvals_rho_A, color='purple')
    plt.yscale('log')
    plt.title('Reduced Density Matrix Spectrum')
    plt.ylabel('Eigenvalue')
    plt.xlabel('Index')
    plt.show()

if __name__ == "__main__":
    main()
